# 15 — Regularized LightGBM validation

This notebook compares the official LightGBM with the best diagnostic regularized candidate on the same development data. It does not load test rows, optimize thresholds, or make a final model decision.

### What this cell does
Maps all existing train/validation, model, robustness, and metadata artifacts; defines new diagnostic outputs; verifies test artifacts only by existence; and fingerprints protected files.

### Why it matters
The comparison must reuse the exact official data and candidate while proving that existing models, preprocessing, reports, and split files are not modified.

### What to understand
Only train and validation content will be loaded. Official test files are acknowledged but never opened.

In [1]:
from pathlib import Path
import hashlib
import json
import os

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from lightgbm import LGBMClassifier
from sklearn.metrics import (average_precision_score, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score)

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
assert ROOT.name == "AdoptAI_V1"
PREPROCESSED_DIR = ROOT / "data/modeling/preprocessed"
REPORT_DIR = ROOT / "reports"
DIAGNOSTIC_MODEL_DIR = ROOT / "models/diagnostic"
FIGURE_DIR = REPORT_DIR / "figures/regularized_lgbm_validation"
DIAGNOSTIC_MODEL_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

paths = {
    "X_train": PREPROCESSED_DIR / "X_train_tree.csv",
    "X_validation": PREPROCESSED_DIR / "X_validation_tree.csv",
    "y_train": PREPROCESSED_DIR / "y_train.csv",
    "y_validation": PREPROCESSED_DIR / "y_validation.csv",
    "validation_identifiers": PREPROCESSED_DIR / "validation_identifiers.csv",
    "official_baseline_lightgbm": ROOT / "models/baseline/lightgbm.joblib",
    "tree_preprocessor": ROOT / "models/preprocessing/tree_preprocessor.joblib",
    "regularized_results": REPORT_DIR / "regularized_lightgbm_results.csv",
    "robustness_by_run": REPORT_DIR / "robustness_by_run.csv",
    "robustness_by_machine": REPORT_DIR / "robustness_by_machine.csv",
    "temporal_scenarios": REPORT_DIR / "temporal_robustness_scenarios.csv",
    "final_split_summary": REPORT_DIR / "final_split_summary.csv",
    "final_split_by_run": REPORT_DIR / "final_split_by_run.csv",
    "feature_names": REPORT_DIR / "preprocessing_feature_report.csv",
}
assert all(path.is_file() for path in paths.values())

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

hashes_before = {name: sha256_file(path) for name, path in paths.items()}
print(f"Existing artifacts mapped and fingerprinted: {len(paths)}")
print("Validation-only boundary established; no test artifact is referenced or loaded.")

Existing artifacts mapped and fingerprinted: 14
Validation-only boundary established; no test artifact is referenced or loaded.


### What this cell does
Loads and validates the official train and validation matrices, labels, identifiers, split metadata, official LightGBM, robustness scenarios, and the exact best regularized candidate row.

### Why it matters
Row alignment and exact parameter recovery are necessary for a fair model-only comparison with no preprocessing or split changes.

### What to understand
The diagnostic candidate combines its saved searched parameters with the same fixed estimator, learning-rate, imbalance, and reproducibility settings as the strong official configuration.

In [2]:
X_train = pd.read_csv(paths["X_train"])
X_validation = pd.read_csv(paths["X_validation"])
y_train = pd.read_csv(paths["y_train"])["slowdown_in_5min"].astype(int)
y_validation = pd.read_csv(paths["y_validation"])["slowdown_in_5min"].astype(int)
validation_identifiers = pd.read_csv(paths["validation_identifiers"])
official_lightgbm = joblib.load(paths["official_baseline_lightgbm"])
regularized_results = pd.read_csv(paths["regularized_results"])
saved_temporal_scenarios = pd.read_csv(paths["temporal_scenarios"])
split_summary = pd.read_csv(paths["final_split_summary"]).set_index("split")
split_by_run = pd.read_csv(paths["final_split_by_run"])

assert X_train.shape == (82_860, 360) and X_validation.shape == (22_212, 360)
assert X_train.columns.tolist() == X_validation.columns.tolist()
assert len(X_train) == len(y_train) and len(X_validation) == len(y_validation) == len(validation_identifiers)
assert not X_train.isna().any().any() and not X_validation.isna().any().any()
assert np.isfinite(X_train.to_numpy(dtype=float)).all() and np.isfinite(X_validation.to_numpy(dtype=float)).all()
official_test_runs = set(split_by_run.loc[split_by_run["split"].eq("test"), "run_id"])
assert official_test_runs.isdisjoint(set(validation_identifiers["run_id"]))

best_candidate_row = regularized_results.loc[regularized_results["experiment"].eq("regularized_random_search")].sort_values("validation_pr_auc", ascending=False).iloc[0]
searched_parameters = json.loads(best_candidate_row["parameters_json"])
train_ratio = float((y_train == 0).sum() / (y_train == 1).sum())
N_JOBS = min(4, max(1, (os.cpu_count() or 2) // 2))
fixed_parameters = {"objective": "binary", "n_estimators": 300, "learning_rate": 0.05,
                    "scale_pos_weight": train_ratio, "subsample_freq": 1,
                    "random_state": 42, "n_jobs": N_JOBS, "verbosity": -1}
regularized_parameters = {**fixed_parameters, **searched_parameters}
print(f"Train positive rate: {y_train.mean():.2%}; validation positive rate: {y_validation.mean():.2%}")
print("No test features, labels, predictions, or test-derived metrics were loaded.")
print("Complete regularized parameters:")
print(json.dumps(regularized_parameters, indent=2, sort_keys=True))

Train positive rate: 34.03%; validation positive rate: 49.13%
No test features, labels, predictions, or test-derived metrics were loaded.
Complete regularized parameters:
{
  "colsample_bytree": 0.9,
  "learning_rate": 0.05,
  "max_depth": 10,
  "min_child_samples": 100,
  "n_estimators": 300,
  "n_jobs": 4,
  "num_leaves": 31,
  "objective": "binary",
  "random_state": 42,
  "reg_alpha": 1.0,
  "reg_lambda": 1.0,
  "scale_pos_weight": 1.938506277040925,
  "subsample": 0.9,
  "subsample_freq": 1,
  "verbosity": -1
}


### What this cell does
Fits the recovered regularized LightGBM on official train only and saves it under a separate diagnostic model path.

### Why it matters
A fresh train-only fit confirms reproducibility while protecting baseline and tuned model files from replacement.

### What to understand
Validation is used only after fitting for comparison, and threshold 0.50 remains fixed.

In [3]:
regularized_lightgbm = LGBMClassifier(**regularized_parameters)
regularized_lightgbm.fit(X_train, y_train)
diagnostic_model_path = DIAGNOSTIC_MODEL_DIR / "regularized_lightgbm.joblib"
diagnostic_parameters_path = DIAGNOSTIC_MODEL_DIR / "regularized_lightgbm_parameters.json"
joblib.dump(regularized_lightgbm, diagnostic_model_path)
diagnostic_parameters_path.write_text(json.dumps(regularized_parameters, indent=2, sort_keys=True) + "\n")
print(f"Saved diagnostic model: {diagnostic_model_path}")
print(f"Saved complete parameters: {diagnostic_parameters_path}")

Saved diagnostic model: /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/models/diagnostic/regularized_lightgbm.joblib
Saved complete parameters: /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/models/diagnostic/regularized_lightgbm_parameters.json


### What this cell does
Calculates train PR-AUC and complete validation metrics for the official and regularized models at threshold 0.50.

### Why it matters
This establishes the global benefit and train–validation gap before investigating whether the gain is stable across runs and machines.

### What to understand
The comparison is model-for-model on identical feature rows; positive deltas favor the regularized model except for errors, where reductions are preferable.

In [4]:
THRESHOLD = 0.50
def metrics(y_true, probability):
    prediction = (probability >= THRESHOLD).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    return {"pr_auc": average_precision_score(y_true, probability),
            "roc_auc": roc_auc_score(y_true, probability) if pd.Series(y_true).nunique() == 2 else np.nan,
            "precision": precision_score(y_true, prediction, zero_division=0),
            "recall": recall_score(y_true, prediction, zero_division=0),
            "f1": f1_score(y_true, prediction, zero_division=0),
            "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}

model_objects = {"official_lightgbm": official_lightgbm, "regularized_lightgbm": regularized_lightgbm}
probabilities, global_rows = {}, []
for model_name, model in model_objects.items():
    train_probability = model.predict_proba(X_train)[:, 1]
    validation_probability = model.predict_proba(X_validation)[:, 1]
    probabilities[model_name] = {"train": train_probability, "validation": validation_probability}
    validation_metrics = metrics(y_validation, validation_probability)
    train_pr_auc = average_precision_score(y_train, train_probability)
    global_rows.append({"model": model_name, "train_pr_auc": train_pr_auc,
                        "validation_pr_auc": validation_metrics["pr_auc"],
                        "train_validation_pr_auc_gap": train_pr_auc-validation_metrics["pr_auc"],
                        **{key: validation_metrics[key] for key in ["roc_auc","precision","recall","f1","tn","fp","fn","tp"]}})
global_comparison = pd.DataFrame(global_rows)
global_comparison_path = REPORT_DIR / "regularized_lgbm_global_comparison.csv"
global_comparison.to_csv(global_comparison_path, index=False)
display(global_comparison)

,model,train_pr_auc,validation_pr_auc,train_validation_pr_auc_gap,roc_auc,precision,recall,f1,tn,fp,fn,tp
0,official_lightgbm,0.999996,0.968988,0.031008,0.958868,0.895394,0.899661,0.897523,10152,1147,1095,9818
1,regularized_lightgbm,0.999960,0.967594,0.032366,0.956799,0.884942,0.897187,0.891022,10026,1273,1122,9791


### What this cell does
Compares both models by validation machine and complete run, adding direct performance deltas and false-negative/false-positive changes.

### Why it matters
Global improvement is only robust if it is not confined to the dominant easy run or obtained by sacrificing the difficult machine.

### What to understand
Positive PR-AUC, recall, and F1 deltas favor regularization; positive FN reduction means fewer missed slowdowns, while positive FP change means more false alerts.

In [5]:
validation_context = validation_identifiers[["machine_id","run_id","segment_id","timestamp"]].copy()
validation_context["true_target"] = y_validation.to_numpy()
for model_name in model_objects:
    validation_context[f"{model_name}_probability"] = probabilities[model_name]["validation"]

def comparison_by(group_columns):
    records=[]; grouping=group_columns[0] if len(group_columns)==1 else group_columns
    for keys,group in validation_context.groupby(grouping,sort=False):
        keys=(keys,) if len(group_columns)==1 else keys
        base={column:value for column,value in zip(group_columns,keys)}
        positives=int(group["true_target"].sum()); negatives=len(group)-positives
        official=metrics(group["true_target"],group["official_lightgbm_probability"])
        regularized=metrics(group["true_target"],group["regularized_lightgbm_probability"])
        record={**base,"rows":len(group),"positive_rate":positives/len(group),"positive_count":positives,"negative_count":negatives}
        for prefix,result in [("official",official),("regularized",regularized)]:
            for key in ["pr_auc","roc_auc","precision","recall","f1","fp","fn"]: record[f"{prefix}_{key}"]=result[key]
        record.update({"delta_pr_auc":regularized["pr_auc"]-official["pr_auc"],
                       "delta_recall":regularized["recall"]-official["recall"],
                       "delta_precision":regularized["precision"]-official["precision"],
                       "delta_f1":regularized["f1"]-official["f1"],
                       "delta_fn":regularized["fn"]-official["fn"],"fn_reduction":official["fn"]-regularized["fn"],
                       "delta_fp":regularized["fp"]-official["fp"],"fp_reduction":official["fp"]-regularized["fp"]})
        records.append(record)
    return pd.DataFrame(records)

by_machine = comparison_by(["machine_id"]); by_run = comparison_by(["machine_id","run_id"])
machine_path=REPORT_DIR/"regularized_lgbm_by_machine.csv"; run_path=REPORT_DIR/"regularized_lgbm_by_run.csv"
by_machine.to_csv(machine_path,index=False); by_run.to_csv(run_path,index=False)
print("By machine:"); display(by_machine)
print("By run:"); display(by_run)

By machine:


/Users/fatimazahranamaoui/Documents/AdoptAI_Project/adoptai_env/lib/python3.13/site-packages/sklearn/metrics/_ranking.py:1192: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/Users/fatimazahranamaoui/Documents/AdoptAI_Project/adoptai_env/lib/python3.13/site-packages/sklearn/metrics/_ranking.py:1192: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/Users/fatimazahranamaoui/Documents/AdoptAI_Project/adoptai_env/lib/python3.13/site-packages/sklearn/metrics/_ranking.py:1192: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/Users/fatimazahranamaoui/Documents/AdoptAI_Project/adoptai_env/lib/python3.13/site-packages/sklearn/metrics/_ranking.py:1192: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


,machine_id,rows,positive_rate,positive_count,negative_count,official_pr_auc,official_roc_auc,official_precision,official_recall,official_f1,...,regularized_fp,regularized_fn,delta_pr_auc,delta_recall,delta_precision,delta_f1,delta_fn,fn_reduction,delta_fp,fp_reduction
0,0890dcc046c079acc4de4202,3024,0.000000,0,3024,0.000000,NaN,0.000000,0.000000,0.000000,...,39,0,0.000000,0.000000,0.000000,0.000000,0,0,-35,35
1,7232bc533c21ce408d45d473,2304,0.383681,884,1420,0.836177,0.845611,0.621008,0.835973,0.712633,...,545,134,0.002911,0.012443,-0.041858,-0.024243,-11,11,94,-94
2,a0f8c86097e55fbfa506d057,8607,0.203555,1752,6855,0.574466,0.746262,0.564121,0.459475,0.506449,...,689,985,-0.026225,-0.021689,-0.037335,-0.028269,38,-38,67,-67
3,d588df123ac0d0ce20b112ac,8277,1.000000,8277,0,1.000000,NaN,1.000000,0.999638,0.999819,...,0,3,0.000000,0.000000,0.000000,0.000000,0,0,0,0


By run:


,machine_id,run_id,rows,positive_rate,positive_count,negative_count,official_pr_auc,official_roc_auc,official_precision,official_recall,...,regularized_fp,regularized_fn,delta_pr_auc,delta_recall,delta_precision,delta_f1,delta_fn,fn_reduction,delta_fp,fp_reduction
0,0890dcc046c079acc4de4202,90c048bb-46c5-4345-a113-d3bc37b876cb,3024,0.000000,0,3024,0.000000,NaN,0.000000,0.000000,...,39,0,0.000000,0.000000,0.000000,0.000000,0,0,-35,35
1,7232bc533c21ce408d45d473,f3c83d3f-0563-4e35-aa47-2e79402b0105,1789,0.263835,472,1317,0.750280,0.817501,0.477215,0.798729,...,495,88,0.025797,0.014831,-0.040355,-0.028997,-7,7,82,-82
2,7232bc533c21ce408d45d473,96f661cc-1233-4539-a814-c4c352953500,247,0.582996,144,103,0.962125,0.944647,0.790055,0.993056,...,50,3,-0.009997,-0.013889,-0.051835,-0.038209,2,-2,12,-12
3,7232bc533c21ce408d45d473,b1dd3935-56dd-4497-a77e-67128988e2c9,268,1.000000,268,0,1.000000,NaN,1.000000,0.817164,...,0,43,0.000000,0.022388,0.000000,0.013395,-6,6,0,0
4,a0f8c86097e55fbfa506d057,0fc9db54-83d3-456a-b9ea-1aa8a6f66cf9,5,1.000000,5,0,1.000000,NaN,1.000000,1.000000,...,0,0,0.000000,0.000000,0.000000,0.000000,0,0,0,0
5,a0f8c86097e55fbfa506d057,baa2a6f4-7121-4bce-9c80-611ebc22ceaf,8602,0.203092,1747,6855,0.572310,0.745536,0.562588,0.457928,...,689,985,-0.026367,-0.021752,-0.037433,-0.028343,38,-38,67,-67
6,d588df123ac0d0ce20b112ac,35472c29-8dd3-49be-8d6b-1984d158745d,8277,1.000000,8277,0,1.000000,NaN,1.000000,0.999638,...,0,3,0.000000,0.000000,0.000000,0.000000,0,0,0,0


### What this cell does
Identifies the lowest-official-PR-AUC validation run, displays its detailed metric change, and summarizes probability distributions by model and true class.

### Why it matters
The difficult run is the central robustness test: improvement there is more informative than a gain driven only by the dominant 85%-positive run.

### What to understand
Probability quartiles describe class separation without selecting any new threshold; classification metrics continue to use 0.50.

In [6]:
informative_runs = by_run.loc[(by_run["rows"] >= 500) & (by_run["positive_count"] > 0) & (by_run["negative_count"] > 0)].copy()
assert not informative_runs.empty
difficult_row = informative_runs.sort_values("official_pr_auc").iloc[0]
difficult_run_id = difficult_row["run_id"]
difficult_mask = validation_context["run_id"].eq(difficult_run_id)
difficult_comparison = pd.DataFrame([
    {"model":"official_lightgbm",**{key:difficult_row[f"official_{key}"] for key in ["pr_auc","precision","recall","f1","fp","fn"]}},
    {"model":"regularized_lightgbm",**{key:difficult_row[f"regularized_{key}"] for key in ["pr_auc","precision","recall","f1","fp","fn"]}},
])
print(f"Difficult run: {difficult_run_id}")
display(difficult_comparison)
print(f"PR-AUC improvement: {difficult_row['delta_pr_auc']:+.6f}")
print(f"Recall improvement: {difficult_row['delta_recall']:+.6f}")
print(f"FN reduction: {int(difficult_row['fn_reduction'])}; FP change: {int(difficult_row['delta_fp']):+d}")

probability_summary_rows=[]
for model_name in model_objects:
    for true_class in [0,1]:
        values=validation_context.loc[difficult_mask & validation_context["true_target"].eq(true_class),f"{model_name}_probability"]
        probability_summary_rows.append({"run_id":difficult_run_id,"model":model_name,"true_class":true_class,"rows":len(values),
                                         "mean_probability":values.mean(),"median_probability":values.median(),
                                         "p25_probability":values.quantile(.25),"p75_probability":values.quantile(.75)})
difficult_probability_summary=pd.DataFrame(probability_summary_rows)
probability_summary_path=REPORT_DIR/"difficult_run_probability_summary.csv"
difficult_probability_summary.to_csv(probability_summary_path,index=False)
display(difficult_probability_summary)

Difficult run: baa2a6f4-7121-4bce-9c80-611ebc22ceaf


,model,pr_auc,precision,recall,f1,fp,fn
0,official_lightgbm,0.572310,0.562588,0.457928,0.504891,622,947
1,regularized_lightgbm,0.545943,0.525155,0.436176,0.476548,689,985


PR-AUC improvement: -0.026367
Recall improvement: -0.021752
FN reduction: -38; FP change: +67


,run_id,model,true_class,rows,mean_probability,median_probability,p25_probability,p75_probability
0,baa2a6f4-7121-4bce-9c80-611ebc22ceaf,official_lightgbm,0,6855,0.209334,0.142208,0.064667,0.295020
1,baa2a6f4-7121-4bce-9c80-611ebc22ceaf,official_lightgbm,1,1747,0.505927,0.425698,0.151710,0.910934
2,baa2a6f4-7121-4bce-9c80-611ebc22ceaf,regularized_lightgbm,0,6855,0.195449,0.110569,0.053079,0.272106
3,baa2a6f4-7121-4bce-9c80-611ebc22ceaf,regularized_lightgbm,1,1747,0.475791,0.368276,0.121171,0.897281


### What this cell does
Reuses the three complete-run scenarios saved by notebook 14 and retrains both the original and regularized configurations on official train for each scenario.

### Why it matters
Reusing identical scenario run IDs prevents favorable resplitting and tests whether regularization consistently changes the same difficult, dominant, and smaller validation runs.

### What to understand
No official test run is present. Each scenario evaluates one entire validation run using fixed threshold 0.50.

In [7]:
temporal_rows=[]
official_clone_parameters=official_lightgbm.get_params(deep=True)
for scenario in saved_temporal_scenarios.itertuples(index=False):
    run_ids=json.loads(scenario.validation_run_ids)
    assert set(run_ids).isdisjoint(official_test_runs)
    mask=validation_identifiers["run_id"].isin(run_ids).to_numpy()
    scenario_results={}
    for name,parameters in [("original",official_clone_parameters),("regularized",regularized_parameters)]:
        model=LGBMClassifier(**parameters); model.fit(X_train,y_train)
        scenario_results[name]=metrics(y_validation.loc[mask],model.predict_proba(X_validation.loc[mask])[:,1])
    temporal_rows.append({"scenario":scenario.scenario,"validation_run_ids":scenario.validation_run_ids,
                          "validation_rows":int(mask.sum()),"validation_positive_rate":float(y_validation.loc[mask].mean()),
                          "machines_represented":scenario.machines_represented,
                          "original_pr_auc":scenario_results["original"]["pr_auc"],"regularized_pr_auc":scenario_results["regularized"]["pr_auc"],
                          "original_recall":scenario_results["original"]["recall"],"regularized_recall":scenario_results["regularized"]["recall"],
                          "original_fn":scenario_results["original"]["fn"],"regularized_fn":scenario_results["regularized"]["fn"],
                          "pr_auc_delta":scenario_results["regularized"]["pr_auc"]-scenario_results["original"]["pr_auc"],
                          "recall_delta":scenario_results["regularized"]["recall"]-scenario_results["original"]["recall"],
                          "fn_reduction":scenario_results["original"]["fn"]-scenario_results["regularized"]["fn"]})
temporal_comparison=pd.DataFrame(temporal_rows)
temporal_comparison_path=REPORT_DIR/"regularized_lgbm_temporal_comparison.csv"
temporal_comparison.to_csv(temporal_comparison_path,index=False)
display(temporal_comparison)

/Users/fatimazahranamaoui/Documents/AdoptAI_Project/adoptai_env/lib/python3.13/site-packages/sklearn/metrics/_ranking.py:1192: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


/Users/fatimazahranamaoui/Documents/AdoptAI_Project/adoptai_env/lib/python3.13/site-packages/sklearn/metrics/_ranking.py:1192: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


,scenario,validation_run_ids,validation_rows,validation_positive_rate,machines_represented,original_pr_auc,regularized_pr_auc,original_recall,regularized_recall,original_fn,regularized_fn,pr_auc_delta,recall_delta,fn_reduction
0,validation_run_1,"[""90c048bb-46c5-4345-a113-d3bc37b876cb""]",3024,0.000000,"[""0890dcc046c079acc4de4202""]",0.000000,0.000000,0.000000,0.000000,0,0,0.000000,0.000000,0
1,validation_run_2,"[""f3c83d3f-0563-4e35-aa47-2e79402b0105""]",1789,0.263835,"[""7232bc533c21ce408d45d473""]",0.750280,0.776077,0.798729,0.813559,95,88,0.025797,0.014831,7
2,validation_run_3,"[""96f661cc-1233-4539-a814-c4c352953500""]",247,0.582996,"[""7232bc533c21ce408d45d473""]",0.962125,0.952128,0.993056,0.979167,1,3,-0.009997,-0.013889,-2
3,validation_run_4,"[""b1dd3935-56dd-4497-a77e-67128988e2c9""]",268,1.000000,"[""7232bc533c21ce408d45d473""]",1.000000,1.000000,0.817164,0.839552,49,43,0.000000,0.022388,6
4,validation_run_5,"[""0fc9db54-83d3-456a-b9ea-1aa8a6f66cf9""]",5,1.000000,"[""a0f8c86097e55fbfa506d057""]",1.000000,1.000000,1.000000,1.000000,0,0,0.000000,0.000000,0
5,validation_run_6,"[""baa2a6f4-7121-4bce-9c80-611ebc22ceaf""]",8602,0.203092,"[""a0f8c86097e55fbfa506d057""]",0.572310,0.545943,0.457928,0.436176,947,985,-0.026367,-0.021752,-38
6,validation_run_7,"[""35472c29-8dd3-49be-8d6b-1984d158745d""]",8277,1.000000,"[""d588df123ac0d0ce20b112ac""]",1.000000,1.000000,0.999638,0.999638,3,3,0.000000,0.000000,0


### What this cell does
Summarizes PR-AUC and recall stability across complete validation runs and applies the predefined robustness decision rule.

### Why it matters
A robust model should improve minimum performance and variability—not merely the dominant global average.

### What to understand
A clear robustness decision requires global performance to be retained, the difficult run to improve, minimum PR-AUC and recall to rise, and PR-AUC variability not to worsen.

In [8]:
stability_rows=[]
for label,prefix in [("official_lightgbm","official"),("regularized_lightgbm","regularized")]:
    pr=informative_runs[f"{prefix}_pr_auc"]; recall=informative_runs[f"{prefix}_recall"]
    stability_rows.append({"model":label,"run_count":len(informative_runs),"mean_pr_auc":pr.mean(),"median_pr_auc":pr.median(),
                           "minimum_pr_auc":pr.min(),"maximum_pr_auc":pr.max(),"std_pr_auc":pr.std(ddof=0),
                           "mean_recall":recall.mean(),"minimum_recall":recall.min(),
                           "total_false_negatives":int(informative_runs[f"{prefix}_fn"].sum()),"total_false_positives":int(informative_runs[f"{prefix}_fp"].sum())})
stability_summary=pd.DataFrame(stability_rows)
stability_path=REPORT_DIR/"regularized_lgbm_stability_summary.csv"
stability_summary.to_csv(stability_path,index=False)
official_global=global_comparison.set_index("model").loc["official_lightgbm"]
regularized_global=global_comparison.set_index("model").loc["regularized_lightgbm"]
official_stability=stability_summary.set_index("model").loc["official_lightgbm"]
regularized_stability=stability_summary.set_index("model").loc["regularized_lightgbm"]
global_retained=regularized_global.validation_pr_auc>=official_global.validation_pr_auc-0.001
difficult_improved=difficult_row.delta_pr_auc>0 and difficult_row.delta_recall>0 and difficult_row.fn_reduction>0
minimums_improved=regularized_stability.minimum_pr_auc>official_stability.minimum_pr_auc and regularized_stability.minimum_recall>official_stability.minimum_recall
variability_not_worse=regularized_stability.std_pr_auc<=official_stability.std_pr_auc+0.001
if global_retained and difficult_improved and minimums_improved and variability_not_worse:
    decision="A. Regularized LightGBM is clearly more robust"
elif regularized_global.validation_pr_auc>official_global.validation_pr_auc and not difficult_improved:
    decision="B. Regularized LightGBM improves global score but not difficult runs"
elif abs(regularized_global.validation_pr_auc-official_global.validation_pr_auc)<0.001 and abs(difficult_row.delta_pr_auc)<0.01:
    decision="C. Regularization gives no meaningful benefit"
else:
    decision="D. Results remain inconclusive because run/machine distribution is too unstable"
display(stability_summary)
print(f"Decision: {decision}")

,model,run_count,mean_pr_auc,median_pr_auc,minimum_pr_auc,maximum_pr_auc,std_pr_auc,mean_recall,minimum_recall,total_false_negatives,total_false_positives
0,official_lightgbm,2,0.661295,0.661295,0.572310,0.750280,0.088985,0.628328,0.457928,1042,1035
1,regularized_lightgbm,2,0.661010,0.661010,0.545943,0.776077,0.115067,0.624868,0.436176,1073,1184


Decision: D. Results remain inconclusive because run/machine distribution is too unstable


### What this cell does
Creates six figures comparing machine, run, difficult-run probability, and temporal-scenario behavior for the two models.

### Why it matters
Visual comparisons make it easier to see whether global gains are broad or concentrated and whether true-class probabilities separate better on the difficult run.

### What to understand
All probability and classification plots use the unchanged validation rows and threshold 0.50; no threshold search is embedded.

In [9]:
figure_paths=[]
def paired_bar(data,index,official,regularized,title,ylabel,filename,ylim=(0,1.03)):
    plot=data.set_index(index)[[official,regularized]].rename(columns={official:"Official",regularized:"Regularized"})
    ax=plot.plot(kind="bar",figsize=(9,5)); ax.set(title=title,ylabel=ylabel,ylim=ylim); ax.grid(axis="y",alpha=.25); plt.xticks(rotation=25,ha="right"); plt.tight_layout(); p=FIGURE_DIR/filename; plt.savefig(p,dpi=160); plt.close(); figure_paths.append(p)
paired_bar(by_machine,"machine_id","official_pr_auc","regularized_pr_auc","PR-AUC by validation machine","PR-AUC","pr_auc_by_machine.png")
paired_bar(by_machine,"machine_id","official_recall","regularized_recall","Recall by validation machine","Recall","recall_by_machine.png")
run_plot=by_run.copy(); run_plot["run_prefix"]=run_plot["run_id"].str[:8]
paired_bar(run_plot,"run_prefix","official_pr_auc","regularized_pr_auc","PR-AUC by complete validation run","PR-AUC","pr_auc_by_run.png")
paired_bar(run_plot,"run_prefix","official_recall","regularized_recall","Recall by complete validation run","Recall","recall_by_run.png")
fig,axes=plt.subplots(2,2,figsize=(12,8),sharex=True,sharey=True)
for row,(model_name,title) in enumerate([("official_lightgbm","Official"),("regularized_lightgbm","Regularized")]):
    for col,true_class in enumerate([0,1]):
        vals=validation_context.loc[difficult_mask & validation_context["true_target"].eq(true_class),f"{model_name}_probability"]
        axes[row,col].hist(vals,bins=30,range=(0,1),alpha=.8); axes[row,col].axvline(.5,color="red",linestyle="--"); axes[row,col].set_title(f"{title}, true class {true_class}"); axes[row,col].set_xlabel("Predicted probability"); axes[row,col].set_ylabel("Rows")
fig.suptitle("Difficult-run probability distributions"); fig.tight_layout(); p=FIGURE_DIR/"difficult_run_probability_distributions.png"; fig.savefig(p,dpi=160); plt.close(fig); figure_paths.append(p)
paired_bar(temporal_comparison,"scenario","original_pr_auc","regularized_pr_auc","Temporal-scenario PR-AUC","PR-AUC","temporal_scenario_comparison.png")
print(f"Saved {len(figure_paths)} figures under {FIGURE_DIR}")

Saved 6 figures under /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/reports/figures/regularized_lgbm_validation


### What this cell does
Verifies every required report, diagnostic artifact, figure, and protected checksum, then prints the requested comparison answers and stop condition.

### Why it matters
The handoff must distinguish a robustness finding from final model selection and prove that test data and official models remained untouched.

### What to understand
The recommendation prioritizes the difficult run and minimum performance; it does not authorize automatic threshold optimization.

In [10]:
required_outputs=[global_comparison_path,machine_path,run_path,temporal_comparison_path,stability_path,probability_summary_path,diagnostic_model_path,diagnostic_parameters_path,*figure_paths]
assert all(path.exists() and path.stat().st_size>0 for path in required_outputs)
assert len(pd.read_csv(machine_path))==validation_identifiers["machine_id"].nunique()
assert len(pd.read_csv(run_path))==validation_identifiers["run_id"].nunique()
assert len(pd.read_csv(temporal_comparison_path))==len(saved_temporal_scenarios)
reloaded_model=joblib.load(diagnostic_model_path); assert isinstance(reloaded_model,LGBMClassifier)
hashes_after={name:sha256_file(path) for name,path in paths.items()}
protected_unchanged=hashes_before==hashes_after; assert protected_unchanged
global_pr_delta=regularized_global.validation_pr_auc-official_global.validation_pr_auc
global_recall_delta=regularized_global.recall-official_global.recall
global_fn_reduction=int(official_global.fn-regularized_global.fn)
temporal_improved_count=int((temporal_comparison["pr_auc_delta"]>0).sum())
print("FINAL REGULARIZED LIGHTGBM VALIDATION REPORT")
print(f"Decision: {decision}")
print(f"Global PR-AUC change: {global_pr_delta:+.6f}")
print(f"Global recall change: {global_recall_delta:+.6f}")
print(f"Global FN reduction: {global_fn_reduction}")
print(f"Difficult run {difficult_run_id}: PR-AUC {difficult_row.official_pr_auc:.6f} -> {difficult_row.regularized_pr_auc:.6f} ({difficult_row.delta_pr_auc:+.6f})")
print(f"Difficult-run recall: {difficult_row.official_recall:.6f} -> {difficult_row.regularized_recall:.6f} ({difficult_row.delta_recall:+.6f}); FN reduction={int(difficult_row.fn_reduction)}, FP change={int(difficult_row.delta_fp):+d}")
print(f"PR-AUC variability: {official_stability.std_pr_auc:.6f} -> {regularized_stability.std_pr_auc:.6f}")
print(f"Minimum PR-AUC: official={official_stability.minimum_pr_auc:.6f}, regularized={regularized_stability.minimum_pr_auc:.6f}")
print(f"Minimum recall: official={official_stability.minimum_recall:.6f}, regularized={regularized_stability.minimum_recall:.6f}")
print(f"Temporal scenarios with PR-AUC improvement: {temporal_improved_count}/{len(temporal_comparison)}")
if decision.startswith("A."):
    print("Recommendation: regularized LightGBM is suitable for a later threshold-optimization comparison, subject to human approval.")
else:
    print("Recommendation: resolve or expand run-level validation coverage before threshold optimization.")
print(f"WARNING: validation remains shifted ({y_validation.mean():.2%} positive versus {y_train.mean():.2%} train).")
print(f"Protected official artifacts unchanged: {protected_unchanged}")
print("STOP: no threshold optimization, calibration, train+validation refit, test evaluation, SHAP, dashboard work, or final model selection was performed.")

FINAL REGULARIZED LIGHTGBM VALIDATION REPORT
Decision: D. Results remain inconclusive because run/machine distribution is too unstable
Global PR-AUC change: -0.001394
Global recall change: -0.002474
Global FN reduction: -27
Difficult run baa2a6f4-7121-4bce-9c80-611ebc22ceaf: PR-AUC 0.572310 -> 0.545943 (-0.026367)
Difficult-run recall: 0.457928 -> 0.436176 (-0.021752); FN reduction=-38, FP change=+67
PR-AUC variability: 0.088985 -> 0.115067
Minimum PR-AUC: official=0.572310, regularized=0.545943
Minimum recall: official=0.457928, regularized=0.436176
Temporal scenarios with PR-AUC improvement: 1/7
Recommendation: resolve or expand run-level validation coverage before threshold optimization.
Protected official artifacts unchanged: True
STOP: no threshold optimization, calibration, train+validation refit, test evaluation, SHAP, dashboard work, or final model selection was performed.
